<div align="center">

# Universidad de San Martín

## Infraestructura para Ciencia de Datos

### Licenciatura en Ciencia de Datos

<img src="../logo.jpg" alt="Logo UNSAM" width="300"/>

---

</div>

# 🏗️ Clase 05: La Bóveda (Capa Gold)

---

## 🎯 Objetivo de Hoy

Modelar la **Capa Gold** (la última de la Arquitectura Medallion) usando Star Schema para BI y Wide Tables (ABT) para ML. La Gold es la capa "lista para consumir": optimizada para queries analíticas, dashboards y modelos de Machine Learning.

---


## 📐 1. Modelado Dimensional: El Star Schema

A diferencia de Silver (donde buscamos integridad y normalización), en Gold buscamos **velocidad y facilidad de uso** para análisis de negocio.

### 🌟 ¿Qué es un Star Schema?

Es un patrón de modelado donde:
- Una **Tabla de Hechos** (Fact Table) está en el centro
- Múltiples **Tablas de Dimensiones** (Dimension Tables) la rodean como rayos de una estrella
- Relaciones simples: cada dimensión se conecta directamente al hecho

**Ventajas:**
- Queries simples y rápidos (menos JOINs)
- Fácil de entender para analistas de negocio
- Optimizado para herramientas BI (Power BI, Tableau, Looker)
- Alto rendimiento en agregaciones

### 📊 Tabla de Hechos (Fact Table)

**Características:**
- Representa una **acción o evento** del negocio (venta, click, envío, transacción)
- Es **larga** (muchas filas) y **estrecha** (pocas columnas)
- Contiene **métricas numéricas** (monto, cantidad, duración)
- Contiene **foreign keys** a las dimensiones
- Crece constantemente (inmutable, solo INSERT)

**Ejemplo: fact_ventas**
```
| venta_id | producto_id | cliente_id | fecha_id | sucursal_id | cantidad | monto | costo | margen |
|----------|-------------|------------|----------|-------------|----------|-------|-------|--------|
| 1001     | 42          | 305        | 20240115 | 7           | 2        | 50.00 | 30.00 | 20.00  |
| 1002     | 15          | 102        | 20240115 | 7           | 1        | 25.00 | 15.00 | 10.00  |
```

**Tipos de Hechos:**
1. **Transaccionales**: Cada fila es un evento (venta, click)
2. **Periódicos (Snapshot)**: Estado en un momento del tiempo (inventario al cierre del día)
3. **Acumulativos**: Proceso con múltiples etapas (pedido → envío → entrega)

### 🎯 Tablas de Dimensiones (Dimension Tables)

**Características:**
- Representan el **contexto** del hecho (quién, qué, dónde, cuándo)
- Son **anchas** (muchas columnas descriptivas) y **cortas** (menos filas)
- Contienen atributos textuales y categóricos
- Se actualizan lentamente (SCD - Slowly Changing Dimensions)
- Permiten filtrar, agrupar y segmentar análisis

**Ejemplo: dim_productos**
```
| producto_id | nombre      | categoria      | subcategoria | marca    | precio_sugerido | activo |
|-------------|-------------|----------------|--------------|----------|-----------------|--------|
| 42          | Mouse Gamer | Periféricos    | Gaming       | Logitech | 25.00           | TRUE   |
| 15          | Teclado USB | Periféricos    | Office       | HP       | 35.00           | TRUE   |
```

**Ejemplo: dim_clientes**
```
| cliente_id | nombre         | email               | ciudad        | segmento | edad | genero |
|------------|----------------|---------------------|---------------|----------|------|--------|
| 305        | Juan Perez     | juan@example.com    | Buenos Aires  | Premium  | 32   | M      |
| 102        | Maria Lopez    | maria@example.com   | Córdoba       | Standard | 28   | F      |
```

**Ejemplo: dim_tiempo** (Dimensión especial)
```
| fecha_id | fecha      | anio | mes | dia | dia_semana | trimestre | es_feriado | mes_nombre |
|----------|------------|------|-----|-----|------------|-----------|------------|------------|
| 20240115 | 2024-01-15 | 2024 | 1   | 15  | Lunes      | Q1        | FALSE      | Enero      |
| 20240116 | 2024-01-16 | 2024 | 1   | 16  | Martes     | Q1        | FALSE      | Enero      |
```

> [!TIP]
> La dimensión tiempo es pre-calculada con TODOS los días del año (incluidos feriados, fines de semana). Esto acelera queries de tendencias temporales.

### 🔗 Star Schema Completo - Diagrama

```mermaid
erDiagram
    FACT_VENTAS ||--o{ DIM_PRODUCTOS : producto_id
    FACT_VENTAS ||--o{ DIM_CLIENTES : cliente_id
    FACT_VENTAS ||--o{ DIM_TIEMPO : fecha_id
    FACT_VENTAS ||--o{ DIM_SUCURSALES : sucursal_id
    
    FACT_VENTAS {
        int venta_id PK
        int producto_id FK
        int cliente_id FK
        int fecha_id FK
        int sucursal_id FK
        int cantidad
        float monto
        float costo
        float margen
    }
    
    DIM_PRODUCTOS {
        int producto_id PK
        string nombre
        string categoria
        string subcategoria
        string marca
        float precio_sugerido
        boolean activo
    }
    
    DIM_CLIENTES {
        int cliente_id PK
        string nombre
        string email
        string ciudad
        string segmento
        int edad
        string genero
    }
    
    DIM_TIEMPO {
        int fecha_id PK
        date fecha
        int anio
        int mes
        int dia
        string dia_semana
        string trimestre
        boolean es_feriado
    }
    
    DIM_SUCURSALES {
        int sucursal_id PK
        string nombre
        string ciudad
        string region
        string tipo
        date fecha_apertura
    }
```

### 📈 Ejemplo de Query de Negocio

**Pregunta:** ¿Cuáles son las ventas totales por categoría y ciudad en Q1 2024?

```sql
SELECT 
    p.categoria,
    c.ciudad,
    t.trimestre,
    SUM(f.monto) as revenue_total,
    COUNT(DISTINCT f.venta_id) as total_transacciones,
    AVG(f.monto) as ticket_promedio
FROM fact_ventas f
JOIN dim_productos p ON f.producto_id = p.producto_id
JOIN dim_clientes c ON f.cliente_id = c.cliente_id
JOIN dim_tiempo t ON f.fecha_id = t.fecha_id
WHERE t.trimestre = 'Q1' AND t.anio = 2024
GROUP BY p.categoria, c.ciudad, t.trimestre
ORDER BY revenue_total DESC;
```

✅ **Solo 3 JOINs** - Simple y rápido para analistas

### 🆚 Star Schema vs Snowflake Schema

| Característica | Star Schema | Snowflake Schema |
|:---|:---|:---|
| **Normalización** | Dimensiones desnormalizadas | Dimensiones normalizadas (sub-dimensiones) |
| **Complejidad** | Simple, menos JOINs | Más complejo, más JOINs |
| **Performance** | Más rápido | Más lento |
| **Espacio** | Más redundancia | Menos redundancia |
| **Uso común** | BI y Reporting | Data Warehouses legacy |

**Recomendación:** Usa **Star Schema** en Gold. El espacio es barato, la simplicidad es cara.

---

## 🔍 1.2 Fact vs Dimension: Diferencias Clave

### 📊 Comparación Completa

| Aspecto | Tabla de Hechos (Fact) | Tabla de Dimensiones (Dim) |
|:---|:---|:---|
| **Contenido** | Eventos/transacciones del negocio | Contexto descriptivo |
| **Columnas** | Métricas numéricas + FKs | Atributos textuales y categóricos |
| **Forma** | Larga (muchas filas) y estrecha (pocas columnas) | Corta (pocas filas) y ancha (muchas columnas) |
| **Tamaño** | Crece continuamente (millones/billones de filas) | Relativamente estable (miles/millones de filas) |
| **Operaciones** | Solo INSERT (inmutable) | INSERT + UPDATE (SCD) |
| **Primary Key** | Generalmente compuesta o surrogate | Simple (ID único) |
| **Foreign Keys** | Muchas (hacia dimensiones) | Ninguna (o hacia otras dimensiones en Snowflake) |
| **Granularidad** | Detalle máximo del evento | Atributos agrupados |
| **Ejemplo** | Una venta específica ($150 el 15/01/2024) | Información del producto (Mouse Gamer, Logitech) |

### 🎯 Cómo Decidir si una Columna va en Fact o Dimension

#### ✅ Va en FACT si:
- Es un **número que se suma** (monto, cantidad, duración)
- Representa una **medida del evento** (temperatura, velocidad, score)
- Cambia en **cada transacción**
- Ejemplo: `monto_venta`, `cantidad_vendida`, `tiempo_entrega`

#### ✅ Va en DIMENSION si:
- Es **descriptivo** (nombre, categoría, color)
- Es **categórico** (región, tipo, estado)
- Es relativamente **estable** en el tiempo
- Se usa para **filtrar o agrupar**
- Ejemplo: `categoria_producto`, `nombre_cliente`, `ciudad_sucursal`

### 🤔 Casos Ambiguos (Degenerate Dimensions)

Algunas columnas son **atributos que no son numéricos pero tampoco justifican una dimensión**:

| Caso | ¿Dónde va? | Razón |
|:---|:---|:---|
| `numero_factura` | **FACT** | Es único por transacción (no se repite) |
| `tipo_pago` (efectivo/tarjeta) | **FACT o DIM** | Pocos valores → puede ir en fact como degenerate dimension |
| `estado_pedido` (pendiente/enviado) | **DIM** | Se usa para filtrar, tiene atributos relacionados |
| `comentarios_cliente` | **❌ Ninguno** | Datos no estructurados → mejor en un data lake |

### 📏 Granularidad de la Fact Table

**Pregunta clave:** ¿Qué representa UNA fila en mi fact table?

**Ejemplos:**

1. **fact_ventas** (granularidad: línea de venta)
   - Cada fila = 1 producto vendido en 1 transacción
   - Si un cliente compra 3 productos → 3 filas

2. **fact_ventas_agregada** (granularidad: transacción)
   - Cada fila = 1 ticket completo
   - Si un cliente compra 3 productos → 1 fila con cantidad=3

3. **fact_inventario_diario** (granularidad: producto x día)
   - Cada fila = Stock de 1 producto al final del día
   - Tipo: Periodic Snapshot

> [!IMPORTANT]
> **Regla de oro:** Define la granularidad ANTES de crear la fact table. Cambiarla después es muy costoso.

### 🔢 Tipos de Métricas en Fact Tables

#### 1. **Additive** (Se pueden sumar en todas las dimensiones)
- `monto_venta` → Suma por producto, cliente, fecha
- `cantidad` → Suma en cualquier nivel

#### 2. **Semi-Additive** (No se suman en todas las dimensiones)
- `saldo_cuenta` → Se suma por clientes, pero NO por fechas (el saldo es un snapshot)
- `inventario_actual` → Similar, no sumar por tiempo

#### 3. **Non-Additive** (Nunca se suman, solo promedios o ratios)
- `temperatura_promedio` → No tiene sentido sumar temperaturas
- `tasa_conversion` → Es un ratio (conversiones / visitas)

```sql
-- Ejemplo: Métrica semi-additive (inventario)
-- ✅ Correcto: Sumar por producto
SELECT producto_id, SUM(inventario_final) as stock_total
FROM fact_inventario_diario
WHERE fecha = '2024-01-15'
GROUP BY producto_id;

-- ❌ Incorrecto: Sumar por fecha (no tiene sentido sumar stocks de días diferentes)
SELECT fecha, SUM(inventario_final) -- ❌ ERROR CONCEPTUAL
FROM fact_inventario_diario
WHERE producto_id = 42
GROUP BY fecha;

-- ✅ Correcto: Promedio o último valor
SELECT fecha, AVG(inventario_final) as promedio_stock
FROM fact_inventario_diario
WHERE producto_id = 42
GROUP BY fecha;
```

---

## 🏢 2. La Capa Semántica (Semantic Layer)

Modelar el **Star Schema** es el primer paso, pero en arquitecturas modernas de datos, el objetivo final es la **Capa Semántica**. Es donde transformamos los datos físicos en conceptos de negocio.

### 🧠 ¿Qué es una Capa Semántica?

Es una abstracción que se sitúa por encima del Star Schema y define las **métricas** y **dimensiones** de forma única para toda la organización. 

**El problema:** Si un analista de marketing calcula el *Revenue* sumando `monto` y otro analista de finanzas lo calcula restando `descuentos`, tenemos dos versiones de la verdad. 

**La solución:** Definir la métrica *Revenue* una sola vez en la Capa Semántica.

| Atributo | Modelado Físico (Star Schema) | Capa Semántica (Semantic Layer) |
|:---|:---|:---|
| **Donde vive** | Tablas de SQL (`fact_ventas`) | Código (dbt Semantic Layer, LookML, MetricStore) |
| **Estructura** | Tablas y Columnas | Métricas y Dimensiones |
| **Fórmula** | Repetida en cada query SQL | Definida una sola vez centralizada |
| **JOINs** | Manuales en cada query | Automáticos según el contexto |

### 🎯 Niveles de Abstracción en Gold

1. **Capa de Modelo (Physical)**: Tablas `fact` y `dim` optimizadas para almacenamiento y JOINs.
2. **Capa Semántica (Logical)**: Definiciones de métricas (ej. `total_revenue = sum(amount)`) y jerarquías (ej. Pais -> Provincia -> Ciudad).
3. **Capa de Presentación (BI)**: Dashboards consumiendo las métricas de la capa lógica.

> [!IMPORTANT]
> **Medallion Gold Standard:** Una buena capa Gold no solo tiene tablas limpias, tiene **métricas gobernadas**. Si cambias la definición de *Margen Bruto*, lo haces en un solo lugar y todos los reportes se actualizan automáticamente.

---

## 🤖 3. ML Gold: La Wide Table (ABT - Analytical Base Table)

A los modelos de Machine Learning no les gusta el Star Schema normalizado. Ellos necesitan una **Wide Table**: una sola tabla gigante donde cada fila es una **entidad** (ej. un Cliente o un Producto).

> [!TIP]
> **Data Engineering for ML** se trata de "aplanar" el modelo dimensional en un set de features (**Feature Engineering**).

### 🎯 ¿Qué es una ABT (Analytical Base Table)?

Una ABT es una tabla **desnormalizada** donde:
- **Cada fila** = 1 entidad de análisis (cliente, producto, transacción)
- **Cada columna** = 1 feature (atributo o métrica agregada)
- **Sin JOINs** = Lista para entrenar modelos (scikit-learn, TensorFlow, PyTorch)

### 📊 Ejemplo: ABT para Predicción de Churn

**Objetivo:** Predecir si un cliente dejará de comprar en los próximos 30 días.

**Features necesarios:**
1. **Demográficos** (de `dim_clientes`): edad, ciudad, segmento
2. **Comportamiento** (agregado de `fact_ventas`):
   - Total de compras
   - Gasto total
   - Frecuencia de compra
3. **Recencia**: Días desde última compra
4. **Target**: ¿El cliente hizo churn? (calculado)

### 🔧 SQL para Crear ABT de Churn

```sql
-- ABT para predicción de Churn (Analytical Base Table)
CREATE TABLE gold_abt_churn AS
WITH ventas_agregadas AS (
    SELECT 
        cliente_id,
        COUNT(venta_id) as total_compras_historico,
        SUM(monto) as gasto_total,
        AVG(monto) as ticket_promedio,
        MAX(fecha) as fecha_ultima_compra,
        MIN(fecha) as fecha_primera_compra,
        COUNT(DISTINCT EXTRACT(MONTH FROM fecha)) as meses_activos
    FROM fact_ventas
    GROUP BY cliente_id
),
recencia AS (
    SELECT 
        cliente_id,
        DATEDIFF('day', MAX(fecha), CURRENT_DATE) as dias_desde_ultima_compra,
        CASE 
            WHEN DATEDIFF('day', MAX(fecha), CURRENT_DATE) > 60 THEN 1 
            ELSE 0 
        END as churn_target  -- Target: 1 = churned, 0 = activo
    FROM fact_ventas
    GROUP BY cliente_id
)
SELECT 
    c.cliente_id,
    -- Features demográficos
    c.edad,
    c.ciudad,
    c.segmento,
    c.genero,
    -- Features de comportamiento
    COALESCE(v.total_compras_historico, 0) as total_compras,
    COALESCE(v.gasto_total, 0) as gasto_total,
    COALESCE(v.ticket_promedio, 0) as ticket_promedio,
    COALESCE(v.meses_activos, 0) as meses_activo,
    -- Features de recencia
    COALESCE(r.dias_desde_ultima_compra, 999) as dias_sin_comprar,
    -- Target (variable a predecir)
    COALESCE(r.churn_target, 1) as churn  -- NULL = nunca compró = churn
FROM dim_clientes c
LEFT JOIN ventas_agregadas v ON c.cliente_id = v.cliente_id
LEFT JOIN recencia r ON c.cliente_id = r.cliente_id;
```

**Resultado (Wide Table):**
```
| cliente_id | edad | ciudad  | segmento | total_compras | gasto_total | ticket_promedio | dias_sin_comprar | churn |
|------------|------|---------|----------|---------------|-------------|-----------------|------------------|-------|
| 101        | 32   | CABA    | Premium  | 45            | 15000.50    | 333.34          | 5                | 0     |
| 102        | 28   | Córdoba | Standard | 12            | 3200.00     | 266.67          | 75               | 1     |
| 103        | 35   | Rosario | Premium  | 0             | 0.00        | 0.00            | 999              | 1     |
```

### 🧪 Uso en Python (Scikit-learn)

```python
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Leer ABT desde Gold
df_abt = pd.read_sql("SELECT * FROM gold_abt_churn", engine)

# Preparar features
# Encodear categorías
le_ciudad = LabelEncoder()
le_segmento = LabelEncoder()

df_abt['ciudad_encoded'] = le_ciudad.fit_transform(df_abt['ciudad'])
df_abt['segmento_encoded'] = le_segmento.fit_transform(df_abt['segmento'])

# Seleccionar features y target
features = ['edad', 'ciudad_encoded', 'segmento_encoded', 
            'total_compras', 'gasto_total', 'ticket_promedio', 'dias_sin_comprar']
X = df_abt[features]
y = df_abt['churn']

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenar modelo
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluar
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy:.2%}")

# Feature importance
importances = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print(importances)
```

### 📋 Tipos Comunes de ABTs

| Tipo ABT | Granularidad | Ejemplo de Uso | Features Típicos |
|:---|:---|:---|:---|
| **Customer ABT** | 1 fila = 1 cliente | Churn, LTV, segmentación | RFM, demográficos, historial compras |
| **Product ABT** | 1 fila = 1 producto | Demand forecasting | Ventas históricas, estacionalidad, categoría |
| **Transaction ABT** | 1 fila = 1 transacción | Fraud detection | Monto, ubicación, hora, frecuencia usuario |
| **Time-Series ABT** | 1 fila = 1 día/hora | Forecasting de ventas | Ventas, temperatura, feriados, promociones |

### 🏗️ ABT vs Star Schema

| Característica | Star Schema (BI) | Wide Table (ML) |
|:---|:---|:---|
| **Propósito** | Reportes y dashboards | Entrenamiento de modelos |
| **Estructura** | Normalizada (fact + dims) | Desnormalizada (flat) |
| **Operación** | Queries con JOINs | Sin JOINs, directo a modelo |
| **Columnas** | Solo lo necesario | Muchas features (100+) |
| **Actualización** | Tiempo real o diario | Batch (semanal/mensual) |
| **Herramientas** | Power BI, Tableau | Scikit-learn, TensorFlow |

### 🎯 Feature Engineering en ABTs

**Tipos de features a agregar:**

1. **Agregaciones numéricas**
   ```sql
   SUM(monto) as gasto_total,
   AVG(monto) as ticket_promedio,
   COUNT(venta_id) as total_transacciones,
   STDDEV(monto) as volatilidad_gasto
   ```

2. **Features temporales**
   ```sql
   DATEDIFF('day', MAX(fecha), CURRENT_DATE) as dias_sin_comprar,  -- Recency
   COUNT(DISTINCT fecha) as frecuencia_compra,                      -- Frequency
   EXTRACT(DOW FROM fecha) as dia_semana_preferido                  -- Patrón temporal
   ```

3. **Ratios y tasas**
   ```sql
   SUM(descuento) / SUM(monto_original) as tasa_descuento_promedio,
   COUNT(devoluciones) / COUNT(compras) as tasa_devolucion
   ```

4. **One-hot encoding de categorías**
   ```sql
   CASE WHEN ciudad = 'Buenos Aires' THEN 1 ELSE 0 END as ciudad_bsas,
   CASE WHEN ciudad = 'Córdoba' THEN 1 ELSE 0 END as ciudad_cordoba
   ```

5. **Features de ventanas temporales** (últimos 30/60/90 días)
   ```sql
   SUM(CASE WHEN fecha > CURRENT_DATE - 30 THEN monto ELSE 0 END) as gasto_ultimos_30d,
   COUNT(CASE WHEN fecha > CURRENT_DATE - 7 THEN 1 END) as compras_ultima_semana
   ```

### ⚡ Best Practices para ABTs

✅ **DO:**
- Incluir `_fecha_creacion` para versioning (ej: `2024-01-15`)
- Mantener múltiples ventanas temporales (7d, 30d, 90d, 1y)
- Documentar cada feature (nombre descriptivo + comentario)
- Guardar en formato Parquet para eficiencia
- Crear índice por la clave principal (cliente_id, producto_id)

❌ **DON'T:**
- Incluir IDs no numéricos sin encodear
- Mezclar datos de entrenamiento con target (data leakage)
- Usar features con > 90% nulos
- Ignorar la distribución de clases (imbalance en target)

---

## 🎓 4. Best Practices para Capa Gold

### ✅ DO - Prácticas Recomendadas

| Práctica | Razón | Ejemplo |
|:---|:---|:---|
| **Crear dimensión de tiempo completa** | Acelera queries de tendencias temporales | Tabla con TODOS los días del año, feriados, trimestres |
| **Usar surrogate keys en dimensiones SCD** | Simplifica JOINs con historia | `sk_cliente` en lugar de `cliente_id` |
| **Desnormalizar en Star Schema** | Queries simples para analistas | Categoría en `dim_productos`, no en tabla separada |
| **Particionar fact tables por fecha** | Mejora performance en queries | `fact_ventas_2024_01`, `fact_ventas_2024_02` |
| **Agregar metadatos de carga** | Facilita debugging y auditoría | `_fecha_carga`, `_proceso_etl_id` |
| **Naming conventions claras** | Facilita comprensión | `fact_*` para hechos, `dim_*` para dimensiones |
| **Documentar KPIs y métricas** | Asegura consistencia entre reportes | Wiki con definición de cada métrica |

### ❌ DON'T - Anti-prácticas

| Anti-Práctica | Problema | Alternativa |
|:---|:---|:---|
| **Normalizar dimensiones (Snowflake)** | Queries complejos, lento | Star Schema desnormalizado |
| **No versionar dimensiones (sin SCD)** | Reportes históricos incorrectos | SCD Tipo 2 para dimensiones que cambian |
| **Fact table sin granularidad clara** | Confusión en análisis | Definir: ¿qué representa 1 fila? |
| **Mezclar BI y ML en misma tabla** | Tabla muy ancha, difícil de mantener | Star Schema para BI, ABT separada para ML |
| **Calcular métricas en BI tool** | Inconsistencias, performance lento | Pre-calcular KPIs en Gold |
| **No particionar tablas grandes** | Queries lentos | Particiones por fecha o región |

### 🏗️ Arquitectura Gold: BI vs ML

```
Capa Silver (Normalizada)
    ├── silver_ventas (fact)
    ├── silver_clientes (dim)
    ├── silver_productos (dim)
    └── silver_sucursales (dim)
           ↓
    ┌──────┴──────┐
    ↓             ↓
Gold BI         Gold ML
(Star Schema)   (Wide Tables)
    ↓             ↓
fact_ventas     gold_abt_churn
dim_clientes    gold_abt_productos
dim_productos   gold_abt_transacciones
dim_tiempo
dim_sucursales
```

### 📊 Estrategia de Particionamiento

**Para Fact Tables grandes (>10M filas):**

```sql
-- Opción 1: Particiones por rango de fecha (PostgreSQL)
CREATE TABLE fact_ventas (
    venta_id INT,
    fecha DATE,
    monto FLOAT,
    ...
) PARTITION BY RANGE (fecha);

CREATE TABLE fact_ventas_2024_q1 PARTITION OF fact_ventas
    FOR VALUES FROM ('2024-01-01') TO ('2024-04-01');

CREATE TABLE fact_ventas_2024_q2 PARTITION OF fact_ventas
    FOR VALUES FROM ('2024-04-01') TO ('2024-07-01');

-- Query solo accede a partición relevante (pruning automático)
SELECT * FROM fact_ventas WHERE fecha BETWEEN '2024-02-01' AND '2024-02-28';
-- Solo lee fact_ventas_2024_q1
```

**Beneficios:**
- Queries 10-100x más rápidos
- Mantenimiento más fácil (archivar particiones antiguas)
- Cargas incrementales eficientes

### 🎯 KPIs: Definición y Documentación

**Crear tabla de metadatos de KPIs:**

```sql
CREATE TABLE gold_kpi_catalog (
    kpi_id INT PRIMARY KEY,
    kpi_name VARCHAR(100),
    kpi_definition TEXT,
    sql_formula TEXT,
    owner VARCHAR(100),
    update_frequency VARCHAR(50),
    created_date DATE
);

-- Ejemplo
INSERT INTO gold_kpi_catalog VALUES (
    1,
    'Revenue Total Mensual',
    'Suma de todas las ventas completadas en el mes calendario',
    'SELECT SUM(monto) FROM fact_ventas WHERE fecha >= DATE_TRUNC(''month'', CURRENT_DATE)',
    'Equipo Analytics',
    'Diario',
    '2024-01-01'
);
```

**Beneficios:**
- Única fuente de verdad para métricas
- Evita definiciones inconsistentes entre equipos
- Facilita onboarding de nuevos analistas

### 🔍 Testing de Calidad en Gold

```sql
-- Test 1: Integridad referencial
SELECT 
    'Fact sin dimensión' as test_name,
    COUNT(*) as filas_huerfanas
FROM fact_ventas f
LEFT JOIN dim_clientes c ON f.sk_cliente = c.sk_cliente
WHERE c.sk_cliente IS NULL;
-- Esperado: 0

-- Test 2: Fact sin duplicados
SELECT 
    'Duplicados en Fact' as test_name,
    COUNT(*) - COUNT(DISTINCT venta_id) as duplicados
FROM fact_ventas;
-- Esperado: 0

-- Test 3: Dimensiones sin nulos en business key
SELECT 
    'Nulos en Business Key' as test_name,
    COUNT(*) as nulos
FROM dim_clientes
WHERE cliente_id IS NULL;
-- Esperado: 0

-- Test 4: SCD consistency
SELECT 
    'Clientes con múltiples registros actuales' as test_name,
    cliente_id,
    COUNT(*) as versiones_actuales
FROM dim_clientes
WHERE es_actual = TRUE
GROUP BY cliente_id
HAVING COUNT(*) > 1;
-- Esperado: 0 resultados
```

### 📈 Monitoreo de Gold Layer

**Métricas clave a trackear:**

```python
gold_metrics = {
    'Volumetría': [
        'Total de filas en fact tables',
        'Crecimiento diario de hechos',
        'Tamaño en disco por tabla'
    ],
    'Performance': [
        'Tiempo promedio de queries en peak',
        'Top 10 queries más lentas',
        'Uso de índices (hit rate)'
    ],
    'Calidad': [
        'Porcentaje de integridad referencial',
        'Número de registros huérfanos',
        'Completitud de dimensiones'
    ],
    'Uso': [
        'Queries ejecutadas por día',
        'Usuarios activos',
        'Tablas más consultadas'
    ]
}
```

### 🚀 Optimizaciones de Performance

| Técnica | Cuándo Usar | Impacto |
|:---|:---|:---|
| **Índices en FK** | Fact tables con JOINs frecuentes | 10-50x más rápido |
| **Índices compuestos** | Filtros multi-columna comunes | 5-20x más rápido |
| **Materialized Views** | Agregaciones costosas y repetidas | 100x más rápido |
| **Columnstore indexes** | Queries analíticos con agregaciones | 10-100x más rápido |
| **Particionamiento** | Fact tables > 10M filas | 10-100x más rápido |
| **Vacuuming/Statistics** | Después de cargas grandes | 2-5x más rápido |

**Ejemplo: Materialized View para KPI frecuente**

```sql
-- Crear vista materializada (pre-calculada)
CREATE MATERIALIZED VIEW gold_mv_revenue_mensual AS
SELECT 
    DATE_TRUNC('month', fecha) as mes,
    p.categoria,
    c.ciudad,
    SUM(f.monto) as revenue_total,
    COUNT(f.venta_id) as total_transacciones
FROM fact_ventas f
JOIN dim_productos p ON f.sk_producto = p.sk_producto
JOIN dim_clientes c ON f.sk_cliente = c.sk_cliente
GROUP BY 1, 2, 3;

-- Crear índice en la vista
CREATE INDEX idx_mv_revenue ON gold_mv_revenue_mensual(mes, categoria, ciudad);

-- Refrescar diariamente (en scheduler)
REFRESH MATERIALIZED VIEW gold_mv_revenue_mensual;

-- Queries ahora son instantáneos
SELECT * FROM gold_mv_revenue_mensual WHERE mes = '2024-01-01';
```

---

## 🐳 5. Pipeline Gold con Airflow — DAGs Pedagógicos

Hasta acá vimos los conceptos: Star Schema, dimensiones, fact tables, ABT, feature engineering. Ahora **bajamos esos conceptos a DAGs reales** sobre datos sintéticos.

| DAG | Genera | Concepto introducido |
|---|---|---|
| `dag_gold_star_basico.py` | `gold.dim_producto` + `gold.dim_tiempo` + `gold.fact_ventas` | Surrogate keys, dimensiones, fact con FKs |
| `dag_gold_abt.py` | `gold.abt_clientes` | ABT (1 fila por cliente), features derivadas, segmentación con `pd.cut` |

Cada cell `%%writefile` escribe el DAG en `stack/dags/03-gold/` automáticamente.

---

### 🐳 DAG 1: Gold Star Schema Básico

Toma `silver.ventas_demo` (datos limpios sintéticos), construye 2 dimensiones (`dim_producto`, `dim_tiempo`) y la `fact_ventas` con las foreign keys correspondientes. Es el patrón clásico de Star Schema para BI.


In [ ]:
%%writefile ../stack/dags/03-gold/dag_gold_star_basico.py

"""
DAG pedagogico: Gold Star Schema Basico
Clase 05 - Construccion de un star schema dimensional.

Pipeline didactico:
  silver.ventas_demo -> dim_producto + dim_tiempo + fact_ventas

Conceptos clave:
  - Surrogate keys (producto_id autogenerado)
  - Dimension tables (1 fila = 1 entidad descriptiva)
  - Fact table (1 fila = 1 evento, con FKs a dimensiones)
"""

from airflow.decorators import dag, task
from datetime import datetime
import os


DB_URI = (
    f"postgresql+psycopg2://"
    f"{os.getenv('SOURCE_DB_USER', 'admin')}:"
    f"{os.getenv('SOURCE_DB_PASS', 'admin')}@"
    f"{os.getenv('SOURCE_DB_HOST', 'data_warehouse')}:5432/"
    f"{os.getenv('SOURCE_DB_NAME', 'InfraCienciaDatos')}"
)


@dag(
    dag_id="gold_star_basico",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
    tags=["gold", "didactico"],
    doc_md="DAG didactico: Star Schema (dim + fact) sobre datos sinteticos.",
)
def gold_star_basico():

    @task
    def prepare_silver():
        """Crea silver.ventas_demo con datos limpios (10 filas, 5 clientes)."""
        import pandas as pd
        import sqlalchemy

        data = {
            "venta_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
            "cliente_id": [101, 102, 103, 101, 104, 105, 102, 103, 101, 104],
            "producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Auriculares",
                         "Laptop", "Mouse", "Teclado", "Auriculares", "Monitor"],
            "precio": [1500.50, 25.00, 75.00, 350.00, 120.00,
                       1450.00, 30.00, 80.00, 110.00, 360.00],
            "cantidad": [1, 2, 1, 1, 3, 1, 4, 1, 2, 1],
            "fecha": ["2024-01-15", "2024-01-16", "2024-01-17", "2024-01-18", "2024-01-19",
                      "2024-02-01", "2024-02-05", "2024-02-10", "2024-03-01", "2024-03-05"],
        }
        df = pd.DataFrame(data)
        df["fecha"] = pd.to_datetime(df["fecha"])

        engine = sqlalchemy.create_engine(DB_URI)
        with engine.begin() as conn:
            conn.execute(sqlalchemy.text("CREATE SCHEMA IF NOT EXISTS silver;"))
        df.to_sql("ventas_demo", engine, schema="silver", if_exists="replace", index=False)
        print(f"silver.ventas_demo: {len(df)} filas listas.")

    @task
    def build_dim_producto():
        """Productos unicos -> gold.dim_producto (con producto_id surrogate)."""
        import pandas as pd
        import sqlalchemy

        engine = sqlalchemy.create_engine(DB_URI)
        df = pd.read_sql(
            "SELECT DISTINCT producto FROM silver.ventas_demo ORDER BY producto",
            engine,
        )
        df["producto_id"] = range(1, len(df) + 1)
        df = df[["producto_id", "producto"]]

        with engine.begin() as conn:
            conn.execute(sqlalchemy.text("CREATE SCHEMA IF NOT EXISTS gold;"))
        df.to_sql("dim_producto", engine, schema="gold", if_exists="replace", index=False)
        print(f"gold.dim_producto: {len(df)} productos.")

    @task
    def build_dim_tiempo():
        """Fechas unicas + atributos temporales -> gold.dim_tiempo."""
        import pandas as pd
        import sqlalchemy

        engine = sqlalchemy.create_engine(DB_URI)
        df = pd.read_sql(
            "SELECT DISTINCT fecha FROM silver.ventas_demo ORDER BY fecha",
            engine,
        )
        df["fecha"] = pd.to_datetime(df["fecha"])
        df["fecha_id"] = df["fecha"].dt.strftime("%Y%m%d").astype(int)
        df["anio"] = df["fecha"].dt.year
        df["mes"] = df["fecha"].dt.month
        df["trimestre"] = df["fecha"].dt.quarter
        df["dia_semana"] = df["fecha"].dt.day_name()
        df = df[["fecha_id", "fecha", "anio", "mes", "trimestre", "dia_semana"]]

        df.to_sql("dim_tiempo", engine, schema="gold", if_exists="replace", index=False)
        print(f"gold.dim_tiempo: {len(df)} fechas.")

    @task
    def build_fact_ventas():
        """Hechos con FKs a dim_producto y dim_tiempo + monto_total derivado."""
        import pandas as pd
        import sqlalchemy

        engine = sqlalchemy.create_engine(DB_URI)
        df = pd.read_sql("SELECT * FROM silver.ventas_demo", engine)
        dim_p = pd.read_sql("SELECT * FROM gold.dim_producto", engine)

        df["fecha"] = pd.to_datetime(df["fecha"])
        df["fecha_id"] = df["fecha"].dt.strftime("%Y%m%d").astype(int)
        df = df.merge(dim_p, on="producto", how="left")
        df["monto_total"] = df["precio"] * df["cantidad"]

        fact = df[["venta_id", "fecha_id", "producto_id", "cliente_id",
                   "precio", "cantidad", "monto_total"]]
        fact.to_sql("fact_ventas", engine, schema="gold", if_exists="replace", index=False)
        print(f"gold.fact_ventas: {len(fact)} filas.")

    silver_done = prepare_silver()
    p = build_dim_producto()
    t = build_dim_tiempo()
    silver_done >> [p, t]
    [p, t] >> build_fact_ventas()


gold_star_basico()


### 🐳 DAG 2: ABT — Wide Table para ML

El Star Schema es ideal para BI (consultas ad-hoc, agregaciones). Pero para Machine Learning preferimos una **wide table**: 1 fila = 1 entidad, N columnas = features.

Este DAG agrupa `silver.ventas_demo` por `cliente_id` y deriva features:

- `total_compras` (count)
- `monto_total` (sum)
- `ticket_promedio` (mean)
- `recencia_dias` (días desde la última compra)
- `segmento_valor` — categórica derivada con `pd.cut`: Bronze / Silver / Gold según monto_total

Y al final una task `verify_integrity` que detecta clientes huérfanos via LEFT JOIN.


In [ ]:
%%writefile ../stack/dags/03-gold/dag_gold_abt.py

"""
DAG pedagogico: Gold ABT (Wide Table) para ML
Clase 05 - Feature Engineering: 1 fila por entidad con metricas agregadas.

Pipeline didactico:
  silver.ventas_demo -> gold.abt_clientes (1 fila por cliente)

Conceptos clave:
  - ABT (Analytical Base Table): tabla ancha, denormalizada
  - Feature engineering: derivar features agregados (count, sum, mean, time-based)
  - Categorizacion con pd.cut (segmentacion por valor)
  - Verificacion de integridad referencial
"""

from airflow.decorators import dag, task
from datetime import datetime
import os


DB_URI = (
    f"postgresql+psycopg2://"
    f"{os.getenv('SOURCE_DB_USER', 'admin')}:"
    f"{os.getenv('SOURCE_DB_PASS', 'admin')}@"
    f"{os.getenv('SOURCE_DB_HOST', 'data_warehouse')}:5432/"
    f"{os.getenv('SOURCE_DB_NAME', 'InfraCienciaDatos')}"
)


@dag(
    dag_id="gold_abt",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
    tags=["gold", "didactico"],
    doc_md="DAG didactico: ABT (wide table) con feature engineering para ML.",
)
def gold_abt():

    @task
    def prepare_silver():
        """Crea silver.ventas_demo (mismo set que gold_star_basico)."""
        import pandas as pd
        import sqlalchemy

        data = {
            "venta_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
            "cliente_id": [101, 102, 103, 101, 104, 105, 102, 103, 101, 104],
            "producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Auriculares",
                         "Laptop", "Mouse", "Teclado", "Auriculares", "Monitor"],
            "precio": [1500.50, 25.00, 75.00, 350.00, 120.00,
                       1450.00, 30.00, 80.00, 110.00, 360.00],
            "cantidad": [1, 2, 1, 1, 3, 1, 4, 1, 2, 1],
            "fecha": ["2024-01-15", "2024-01-16", "2024-01-17", "2024-01-18", "2024-01-19",
                      "2024-02-01", "2024-02-05", "2024-02-10", "2024-03-01", "2024-03-05"],
        }
        df = pd.DataFrame(data)
        df["fecha"] = pd.to_datetime(df["fecha"])

        engine = sqlalchemy.create_engine(DB_URI)
        with engine.begin() as conn:
            conn.execute(sqlalchemy.text("CREATE SCHEMA IF NOT EXISTS silver;"))
        df.to_sql("ventas_demo", engine, schema="silver", if_exists="replace", index=False)

    @task
    def build_abt_clientes():
        """Agrupa por cliente_id y deriva features de comportamiento."""
        import pandas as pd
        import sqlalchemy

        engine = sqlalchemy.create_engine(DB_URI)
        df = pd.read_sql("SELECT * FROM silver.ventas_demo", engine)
        df["fecha"] = pd.to_datetime(df["fecha"])
        df["monto"] = df["precio"] * df["cantidad"]

        ahora = pd.Timestamp.now().normalize()

        abt = df.groupby("cliente_id").agg(
            total_compras=("venta_id", "count"),
            monto_total=("monto", "sum"),
            ticket_promedio=("monto", "mean"),
            ultima_compra=("fecha", "max"),
            primera_compra=("fecha", "min"),
        ).reset_index()

        abt["recencia_dias"] = (ahora - abt["ultima_compra"]).dt.days
        abt["ticket_promedio"] = abt["ticket_promedio"].round(2)

        # Segmentacion por monto_total (Bronze / Silver / Gold)
        abt["segmento_valor"] = pd.cut(
            abt["monto_total"],
            bins=[0, 100, 1000, float("inf")],
            labels=["Bronze", "Silver", "Gold"],
        ).astype(str)

        with engine.begin() as conn:
            conn.execute(sqlalchemy.text("CREATE SCHEMA IF NOT EXISTS gold;"))
        abt.to_sql("abt_clientes", engine, schema="gold", if_exists="replace", index=False)
        print(f"gold.abt_clientes: {len(abt)} clientes con features.")

    @task
    def verify_integrity():
        """LEFT JOIN para detectar clientes huerfanos."""
        import pandas as pd
        import sqlalchemy

        engine = sqlalchemy.create_engine(DB_URI)
        q = """
            SELECT COUNT(DISTINCT v.cliente_id) AS huerfanos
            FROM silver.ventas_demo v
            LEFT JOIN gold.abt_clientes a ON v.cliente_id = a.cliente_id
            WHERE a.cliente_id IS NULL
        """
        n = int(pd.read_sql(q, engine).iloc[0, 0])
        if n > 0:
            print(f"WARNING: {n} clientes huerfanos detectados.")
        else:
            print("OK: integridad referencial verificada.")

    silver_done = prepare_silver()
    abt_done = build_abt_clientes()
    silver_done >> abt_done >> verify_integrity()


gold_abt()


## 📊 6. Visualización — Dashboard sobre Gold

Los datos en Gold sirven a dos audiencias: **BI** (Star Schema, queries SQL) y **ML** (ABT, features). Esta sección genera una **página de Streamlit** que muestra ambos.

El siguiente cell escribe una nueva página en `stack/dashboard/pages/7_Demo_Pedagogico.py`. Streamlit la detecta automáticamente y aparece en el menú lateral de `localhost:8501`.

**Qué visualiza:**
- Métricas generales (productos, fechas, ventas, clientes)
- Ventas por producto (JOIN fact_ventas + dim_producto)
- Ventas por mes (JOIN fact_ventas + dim_tiempo)
- Segmentación de clientes (ABT con `segmento_valor`)

> **Pre-requisito**: corré primero los DAGs `gold_star_basico` y `gold_abt` para que las tablas existan.


In [ ]:
%%writefile ../stack/dashboard/pages/7_Demo_Pedagogico.py

"""
Pagina demo pedagogica - Visualiza el Star Schema y ABT generados por los
DAGs didacticos de clase05 (gold_star_basico, gold_abt).

Conecta con las tablas:
  - gold.dim_producto, gold.dim_tiempo, gold.fact_ventas (Star Schema)
  - gold.abt_clientes (ABT con features)
"""

import streamlit as st
import plotly.express as px
import pandas as pd
from db import load_table, get_row_count, table_exists, get_engine

st.header("Demo Pedagogico - Star Schema y ABT")
st.caption("Vista de los datos sinteticos generados por los DAGs `gold_star_basico` y `gold_abt`.")

engine = get_engine()
needed = [("gold", "fact_ventas"), ("gold", "dim_producto"),
          ("gold", "dim_tiempo"), ("gold", "abt_clientes")]
missing = [f"{s}.{t}" for s, t in needed if not table_exists(engine, s, t)]

if missing:
    st.warning(
        "Faltan tablas: " + ", ".join(missing) +
        ". Ejecuta los DAGs `gold_star_basico` y `gold_abt` desde Airflow."
    )
    st.stop()

# =============================================================
# METRICAS GENERALES
# =============================================================
col1, col2, col3, col4 = st.columns(4)
col1.metric("Productos", get_row_count("gold", "dim_producto"))
col2.metric("Fechas", get_row_count("gold", "dim_tiempo"))
col3.metric("Ventas", get_row_count("gold", "fact_ventas"))
col4.metric("Clientes", get_row_count("gold", "abt_clientes"))

st.divider()

# =============================================================
# STAR SCHEMA: Ventas por producto
# =============================================================
st.subheader("Ventas por producto (Star Schema)")
st.caption("Join: fact_ventas + dim_producto. Suma del monto total por producto.")

q_ventas_producto = """
    SELECT p.producto, SUM(f.monto_total) AS total
    FROM gold.fact_ventas f
    JOIN gold.dim_producto p ON f.producto_id = p.producto_id
    GROUP BY p.producto
    ORDER BY total DESC
"""
df_vp = pd.read_sql(q_ventas_producto, engine)

if not df_vp.empty:
    fig = px.bar(df_vp, x="producto", y="total",
                 color="total", color_continuous_scale="Blues",
                 labels={"total": "Monto total ($)", "producto": ""})
    fig.update_layout(height=350, showlegend=False)
    st.plotly_chart(fig, use_container_width=True)

st.divider()

# =============================================================
# STAR SCHEMA: Ventas por mes
# =============================================================
st.subheader("Ventas por mes")
st.caption("Join: fact_ventas + dim_tiempo. Cantidad de ventas por mes.")

q_ventas_mes = """
    SELECT t.anio, t.mes, COUNT(*) AS ventas, SUM(f.monto_total) AS monto
    FROM gold.fact_ventas f
    JOIN gold.dim_tiempo t ON f.fecha_id = t.fecha_id
    GROUP BY t.anio, t.mes
    ORDER BY t.anio, t.mes
"""
df_vm = pd.read_sql(q_ventas_mes, engine)
if not df_vm.empty:
    df_vm["periodo"] = df_vm["anio"].astype(str) + "-" + df_vm["mes"].astype(str).str.zfill(2)
    fig2 = px.line(df_vm, x="periodo", y="monto", markers=True,
                   labels={"monto": "Monto total ($)", "periodo": ""})
    fig2.update_layout(height=300)
    st.plotly_chart(fig2, use_container_width=True)

st.divider()

# =============================================================
# ABT: Segmentacion de clientes
# =============================================================
st.subheader("ABT - Segmentacion de clientes por valor")
st.caption("1 fila = 1 cliente. Features derivadas (total_compras, monto_total, ticket_promedio, recencia, segmento).")

abt = load_table("gold", "abt_clientes")
if not abt.empty:
    col_a, col_b = st.columns([1, 2])

    with col_a:
        seg_count = abt["segmento_valor"].value_counts().reset_index()
        seg_count.columns = ["segmento", "clientes"]
        fig_seg = px.pie(seg_count, names="segmento", values="clientes", hole=0.4,
                         color="segmento",
                         color_discrete_map={"Bronze": "#cd7f32",
                                              "Silver": "#c0c0c0",
                                              "Gold": "#ffd700"})
        fig_seg.update_layout(height=280, margin=dict(t=10, b=10))
        st.plotly_chart(fig_seg, use_container_width=True)

    with col_b:
        st.dataframe(
            abt.sort_values("monto_total", ascending=False),
            use_container_width=True, hide_index=True,
        )

st.divider()
st.caption(
    "**Patron didactico:** los DAGs `gold_star_basico` y `gold_abt` generan "
    "estas tablas con datos sinteticos hardcoded. En el DAG productivo "
    "`crypto_gold` el mismo patron se aplica sobre datos reales de criptomonedas."
)


---

## 🎉 ¡Capa Gold completada!

Acabás de construir un Star Schema (BI-friendly) y una ABT (ML-friendly), y los visualizaste en un dashboard. Esto **cierra el pipeline Medallion** — Bronze → Silver → Gold → Consumo.

**Próximo paso (opcional)**: aplicá lo aprendido sobre datos reales de criptomonedas en el [Ejercicio de la Clase 05](ejercicios/ejercicio.ipynb).
